# 214 — Concatenated clustering: response TIMING and cluster ordering

Stage-04 pooling asks *"when does each a-priori functional **role** come on, and in what
order?"* and answers it with a mean-HGA time course per role plus an onset ladder
(status site FIG 4.2 / 4.3). This notebook asks exactly the same question of the
**data-driven clusters**:

> **when does each concatenated cluster's high-gamma response come on, and in what order?**

The difference matters. Roles are hypothesis-driven and fixed, so their ordering is a
**test of prior theory**. Clusters are discovered, so their ordering is a **description of
what the data actually separates**. If the two ladders agree, the a-priori roles are
recovering real timing structure rather than imposing it.

**Why the concatenated track specifically.** One sample = one electrode carrying
`[audio | picture | reading]`, so a cluster's three onsets belong to the *same electrodes*
and are directly comparable to each other — which is not true of the per-condition tracks,
where an "audio cluster" and a "reading cluster" contain different contacts.

### Onset definition — deliberately identical to pooling

    onset = first bin whose |mean HGA| exceeds `ONSET_THR_DB`,
            expressed as % of the warped trial (50% = GO cue).
            No crossing anywhere -> 100.0, i.e. "never".

Keeping it byte-identical to `lf_pool.plot_role_hga_timeseries` is the whole point: it is
what lets the cluster ladder and the role ladder be read on the same axis.

Two consequences worth holding in mind:

* it uses **|mean|**, so a strongly *suppressed* cluster registers an onset just as an
  activated one does. Check the sign in `peak_db` before calling something "early activity".
* onset is a **threshold crossing**, so it is sensitive to cluster size — a large,
  heterogeneous cluster averages toward zero and crosses late or never.

### Every K, not just the winner

The whole sweep is walked, so you can see whether the ordering is a property of the data
or of the K we happened to pick. An ordering that survives K=4..20 is the former.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

from functions import lf_cluster_timing as T

SCRIPT_NAME = '214_concat_timingranking.ipynb'
pd.set_option('display.width', 160)

## 1. Pick the run

Any run with a `cluster_labels_by_k.csv` works — that file is written by every K-sweep
fit, so no re-clustering happens here. `concat_hg` is the natural target: it is already
the HG line (70–150 Hz mean) stitched across the three conditions, so there is no
frequency axis left to collapse and the feature matrix *is* the time course.

In [ ]:
CLUSTERING = Path('outputs/clustering')

METHOD      = 'kmeans'
FEATURE_SET = 'concat_hg'      # concat_rawds also works — see N_BLOCKS note below
RUN_ID      = None             # None -> newest run in that track

# Onset: identical default to lf_pool. 3 condition blocks in the concatenated sample.
ONSET_THR_DB = 0.5
N_BLOCKS     = 3
CONDITIONS   = ('audio', 'picture', 'reading')

track = CLUSTERING / METHOD / FEATURE_SET / 'runs'
runs  = sorted(d for d in track.iterdir() if d.is_dir()) if track.is_dir() else []
if not runs:
    raise SystemExit(f'no runs under {track} — run 233 first')
RUN_DIR = (track / RUN_ID) if RUN_ID else runs[-1]
print('run      :', RUN_DIR)

X_hg = np.load(RUN_DIR / 'X_train.npy')
labels_by_k = T.load_labels_by_k(RUN_DIR)
print('X_train  :', X_hg.shape, ' ->', T.block_view(X_hg, N_BLOCKS).shape,
      '(n, blocks, time)')
print('K sweep  :', sorted(labels_by_k))

## 2. Timing table for every K

One row per (K, cluster, condition): onset, peak latency, peak amplitude, mean dB and
post-GO AUC. This is the sortable object — everything below is a view of it.

`crosses=False` means the cluster never exceeded the threshold in that condition. That is
a real result ("silent here"), not missing data, so `onset_pct` is kept as 100.0 and it
simply sorts last.

In [ ]:
TIMING_DIR = RUN_DIR / 'timing'
tab = T.sweep_timing(X_hg, labels_by_k, n_blocks=N_BLOCKS, conditions=CONDITIONS,
                     onset_thr_db=ONSET_THR_DB, out_dir=TIMING_DIR)

print(f'{len(tab)} rows over {tab["k"].nunique()} K values -> {TIMING_DIR}')
print()
print('median onset by condition, pooled over every K and cluster:')
print(tab.groupby('condition')['onset_pct'].median().round(1).to_string())
print()
never = tab[~tab['crosses']]
print(f'cluster x condition cells that never cross: {len(never)} / {len(tab)}')
print(never.groupby('condition').size().to_string())

## 3. The chosen K — time courses and the ladder

The cluster analogue of pooling's **FIG 4.2** (mean HGA per role, one panel per condition)
and **FIG 4.3** (the onset ladder).

In the ladder, clusters are ordered by their **across-condition mean onset**, and that same
order is used in all three panels — if each panel re-sorted itself, the comparison the
figure exists to support would be impossible to make.

In [ ]:
# K_MAIN = int(pd.read_json(RUN_DIR / 'manifest.json', typ='series').get('n_clusters')
#              or sorted(labels_by_k)[0])
# print('main K =', K_MAIN)

# T.plot_cluster_hga_timeseries(X_hg, labels_by_k[K_MAIN], n_blocks=N_BLOCKS,
#                               conditions=CONDITIONS, onset_thr_db=ONSET_THR_DB,
#                               title=f'K={K_MAIN}')
# plt.show()
KS_TO_PLOT = sorted(labels_by_k)          # or e.g. [4, 5, 6, 8, 10, 15, 20]


for k in KS_TO_PLOT:
    T.plot_cluster_hga_timeseries(X_hg, labels_by_k[k], n_blocks=N_BLOCKS,
                                  conditions=CONDITIONS, onset_thr_db=ONSET_THR_DB,
                                  title=f'K={k}')
    plt.show()
    T.plot_onset_ladder(tab[tab['k'] == k], title=f'Onset ordering · K={k}')
    plt.show()

In [ ]:
# T.plot_onset_ladder(tab[tab['k'] == K_MAIN], title=f'Onset ordering · K={K_MAIN}')
# plt.show()

# print('onset matrix (sorted by mean onset across conditions):')
# T.onset_matrix(tab, K_MAIN).round(1)

## 4. Is the ordering stable across K?

Each dot is one cluster's onset at one K, sized by cluster n; the black line is the median.

What to look for: if the **spread keeps its shape** as K grows, the timing structure is a
property of the data and finer partitions are subdividing it rather than inventing it. If
the spread collapses or scatters, the ordering only exists at particular K and should not
be reported as a finding.

In [ ]:
T.plot_onset_across_k(tab)
plt.show()

# Spread of onsets within each K — a proxy for "how much timing structure is there".
spread = (tab.groupby(['k', 'condition'])['onset_pct']
            .agg(['median', 'std', 'min', 'max']).round(1))
spread.unstack('condition')['std']

## 5. Compare against the pooling role ladder

The direct test of whether the discovered clusters and the a-priori roles describe the same
timing structure. Agreement would be meaningful precisely because nothing is shared: the
roles were written from prior anatomy/physiology, the clusters were found without labels.

**This needs one thing that does not exist yet.** `460` computes the role onsets in memory
(`plot_role_hga_timeseries` returns `onset_dict`) and uses them to draw
`role_onset_ordering.png`, but never writes them down — `role_info.json` has no `onset`
field. Its own docstring says *"Patch onset_dict into role_info.json"*; that was never done.

To enable this cell, add one line to 460 after the `plot_role_hga_timeseries` call:

```python
fig, onset_dict = LP.plot_role_hga_timeseries(...)
for role, onsets in onset_dict.items():
    role_info[role]['onset'] = onsets          # <- persist it
LP.write_json(POOL_WEB / 'role_info.json', role_info)
```

Until then this cell reports what is missing and stops, rather than silently comparing
against nothing.

In [ ]:
import json

role_json = Path('../04_FBM_Pooling/outputs/pooling/pool_web/role_info.json')
MISSING = ("role_info.json carries no `onset` field. 460 computes the role onsets but "
           "never persists them - see the note above for the 3-line patch, then re-run 460.")

roles = pd.DataFrame()
if not role_json.exists():
    print('role_info.json not found - run 460 first.')
else:
    ri = json.loads(role_json.read_text(encoding='utf-8'))
    rows = [{'name': r, 'condition': c, 'onset_pct': v}
            for r, d in ri.items()
            for c, v in (d.get('onset') or {}).items()
            if isinstance(v, (int, float))]
    roles = pd.DataFrame(rows)
    if roles.empty:
        print(MISSING)

if not roles.empty:
    clus = (tab[tab['k'] == K_MAIN][['cluster', 'condition', 'onset_pct']]
            .rename(columns={'cluster': 'name'}))
    clus['name'] = 'c' + clus['name'].astype(str)
    both = pd.concat([clus.assign(kind='cluster'), roles.assign(kind='role')])

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True)
    for ax, cond in zip(np.atleast_1d(axes), CONDITIONS):
        s = both[both['condition'] == cond].sort_values('onset_pct')
        colors = ['#b85c6e' if k == 'cluster' else '#0b7a75' for k in s['kind']]
        ax.barh(range(len(s)), s['onset_pct'], color=colors, height=.72)
        ax.set_yticks(range(len(s)))
        ax.set_yticklabels(s['name'], fontsize=7.5)
        ax.invert_yaxis()
        ax.axvline(50, color='0.4', ls='--', lw=1)
        ax.set_title(cond.capitalize(), fontsize=10.5)
        ax.set_xlabel('onset (%)')
        for sp in ('top', 'right', 'left'):
            ax.spines[sp].set_visible(False)
    fig.suptitle('Onset ordering - clusters (red) vs a-priori roles (teal)',
                 fontsize=11.5, y=1.02)
    fig.tight_layout()
    plt.show()


## 6. What was written

    <run>/timing/
      cluster_timing_by_k.csv     every (K, cluster, condition) row — the sortable table
      timing_k<NN>_hga.png        mean HGA per cluster, per condition, one file per K
      timing_k<NN>_onset.png      the onset ladder, one file per K
      onset_across_k.png          every cluster's onset at every K

**Reading these honestly.** Onset is a first threshold crossing on a *time-warped* axis, so
the ordering is the result and the exact percentages are not. Two clusters four points
apart are not meaningfully different; one at 10% and one at 68% are. And check `peak_db`
before describing a cluster as early-active — the |mean| criterion treats suppression and
activation alike.

---

## Next: cross-correlation

*When* each cluster comes on is answered above. Whether the clusters **lead and lag each other** is a separate question with its own pitfalls, so it lives in its own notebook: **`215_concat_crosscorrelation.ipynb`**.

It reuses `lf_cluster_timing` and the same run, and it needs the onset table this notebook produces (`timing/cluster_timing_by_k.csv`) to pick its reference cluster.